# WaferPulse - Exploratory Telemetry & Chamber Anomaly Analysis

This notebook demonstrates exploratory signal diagnostics on multi-sensor wafer telemetry batches:
1. **Telemetry Ingestion & Schema Inspection**
2. **Sensor Missingness & Zero-Variance Diagnostics (Dead Sensors)**
3. **Chamber Regime Clustering (K-Means with Silhouette Optimization)**
4. **Sensor Anomaly & Z-Score Defect Attribution**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add repository root to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data.generator import generate_wafer_telemetry
from src.features.preprocessor import TelemetryPreprocessor
from src.models.clusterer import WaferClusterer
from src.explain.root_cause import diagnose_single_wafer

## 1. Generate & Inspect Raw Telemetry Batch

In [ ]:
df = generate_wafer_telemetry(num_samples=300, num_sensors=60, defect_rate=0.08, random_state=42)
print(f"Raw telemetry shape: {df.shape}")
print(f"Target distribution: {df['Output'].value_counts().to_dict()}")
df.head()

## 2. Zero-Variance & Dead Sensor Detection
Certain sensor channels in fabrication tools become disconnected or uncalibrated (zero variance across runs).

In [ ]:
sensor_cols = [c for c in df.columns if c.startswith('Sensor_')]
stds = df[sensor_cols].std()
zero_var_sensors = stds[stds == 0].index.tolist()
print(f"Pruned dead sensors (zero variance): {zero_var_sensors}")

## 3. Chamber Regime Clustering
Wafers operate across distinct physical chamber regimes (e.g. CMP, Lithography, Plasma Etch). We use K-Means to partition them into operational regimes.

In [ ]:
preprocessor = TelemetryPreprocessor()
X, y, wafer_ids = preprocessor.fit_transform(df)

clusterer = WaferClusterer(min_clusters=2, max_clusters=4, random_state=42)
cluster_labels = clusterer.fit_predict(X)

print(f"Optimal Chamber Clusters (K): {clusterer.optimal_clusters}")
print(f"Silhouette scores: {clusterer.silhouette_scores}")
print(f"Distribution across clusters: {clusterer.cluster_distribution}")

## 4. Root Cause Sensor Anomaly Attribution
When a wafer fails quality control, which specific chamber sensor caused the deviation?

In [ ]:
# Find defective wafer
defect_idx = np.where(y == -1)[0][0]
defective_row = df.iloc[defect_idx].to_dict()

diagnosis = diagnose_single_wafer(
    wafer_id=wafer_ids[defect_idx],
    sensor_values=defective_row,
    baseline_stats=preprocessor.baseline_stats,
    defect_prob=0.98,
    cluster_id=cluster_labels[defect_idx],
    top_k=3
)

print(f"Diagnosis for {diagnosis.wafer_id} (Cluster {diagnosis.cluster_id}):")
for a in diagnosis.top_anomalies:
    print(f"  -> {a.sensor}: value={a.value:.1f} vs mean={a.baseline_mean:.1f} ({a.direction}, Z={a.z_score:.2f} sigma) [{a.severity}]")